In [2]:
import matplotlib.pyplot as plt
import scienceplots
import polars as pl
from pygments.styles.dracula import background

plt.style.use('science')

INPUT_FILE = "../protbert/datasets/dataset_merged_with_families.parquet"

df = pl.read_parquet(INPUT_FILE).filter(pl.col("reverse") == False)
df

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,cath_dominant,cath_all,cath_class,cath_arch,cath_topology,cath_homology
str,str,str,f64,bool,str,str,str,str,str,str,str
"""SAGNQESVVAAVLIPINTALTVGMMTTRVV…","""SAGNQESVVAAVLIPINTALTVGMMTTRVV…","""P68Y""",-0.295097,false,"""megascale""","""G3DSA:3.90.1750.10""","""G3DSA:2.20.70.10;G3DSA:2.60.40…","""G3DSA:3""","""90""","""1750""","""10"""
"""SAGGSAGGKFNKELSVAGREIVTLPNLNDP…","""SAGGSAGGKFNKELSVAGREIVTLPNLNDP…","""D41Y""",-0.018312,false,"""megascale""","""G3DSA:2.30.42.10""","""G3DSA:2.20.70.10;G3DSA:2.30.42…","""G3DSA:2""","""30""","""42""","""10"""
"""SAGKMTGIVKWFNADKGFGFITPDDGSKDV…","""SAGKMTGIVKWFNADKGFGFITPDDGSKDV…","""Q38L""",0.046215,false,"""megascale""","""G3DSA:3.30.170.10""","""G3DSA:3.30.170.10""","""G3DSA:3""","""30""","""170""","""10"""
"""GTTVKVNGTKYKFDTPEEAQKFAKKAADKY…","""GTTVKVNGTKYKFDTPEEAQKFAKKAADKY…","""R42T""",-0.029416,false,"""megascale""","""G3DSA:3.30.890.10""","""G3DSA:3.30.890.10""","""G3DSA:3""","""30""","""890""","""10"""
"""SAGGSAGGSAGGDEIHIHFNGVTIEFRGLT…","""SAGGSAGGSAGGDEIHIHFNGVTIEFRGLT…","""E47K""",-0.023229,false,"""megascale""","""G3DSA:1.20.5.420""","""G3DSA:1.20.5.420""","""G3DSA:1""","""20""","""5""","""420"""
…,…,…,…,…,…,…,…,…,…,…,…
"""MATLNSASTTGTTPSPGHNAPSLPSDTFSS…","""MATLNSASTTGTTPSPGHNAPSLPSDTFSS…","""S1859R""",-0.095211,false,"""lehner""","""G3DSA:2.30.42.10""","""G3DSA:1.10.287.650;G3DSA:2.30.…","""G3DSA:2""","""30""","""42""","""10"""
"""MATLNSASTTGTTPSPGHNAPSLPSDTFSS…","""MATLNSASTTGTTPSPGHNAPSLPSDTFSS…","""T1859R""",-0.367719,false,"""lehner""","""G3DSA:2.30.42.10""","""G3DSA:1.10.287.650;G3DSA:2.30.…","""G3DSA:2""","""30""","""42""","""10"""
"""MATLNSASTTGTTPSPGHNAPSLPSDTFSS…","""MATLNSASTTGTTPSPGHNAPSLPSDTFSS…","""V1859R""",-0.345122,false,"""lehner""","""G3DSA:2.30.42.10""","""G3DSA:1.10.287.650;G3DSA:2.30.…","""G3DSA:2""","""30""","""42""","""10"""


In [3]:
import plotly.express as px

hierarchy_df = (
    df.group_by(["cath_class", "cath_arch", "cath_topology", "cath_homology"])
    .agg(pl.len().alias("count"))
    .to_pandas() # Plotly zatím vyžaduje pandas nebo dict, polars neumí napřímo
)

# 3. Vykreslení Sunburst diagramu (interaktivní strom)
fig = px.sunburst(
    hierarchy_df,
    path=['cath_class', 'cath_arch', 'cath_topology', 'cath_homology'],
    values='count',
    title="Hierarchie CATH rodin a počty sekvencí",
    color='cath_class',
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_traces(textinfo="label+value")
fig.show()

# 4. Pokud preferuješ klasický Treemap (čtvercový strom)
fig_tree = px.treemap(
    hierarchy_df,
    path=['cath_class', 'cath_arch', 'cath_topology', 'cath_homology'],
    values='count',
    title="Treemap CATH struktur",
    color='count',
    color_continuous_scale='Viridis'
)
fig = px.treemap(
    hierarchy_df,
    path=['cath_class', 'cath_arch', 'cath_topology', 'cath_homology'],
    values='count'
)

# Export přímo do PDF
fig.write_image("cath_hierarchy.pdf")

In [5]:
import polars as pl
import plotly.express as px


# 2. Agregace pro vizualizaci
# CATH úrovně: Class, Architecture, Topology, Homology
hierarchy_df = (
    df.group_by(["cath_class", "cath_arch", "cath_topology", "cath_homology"])
    .len()
    .to_pandas()
)

# 3. Tvorba Sunburst grafu (vypadá nejvíce "vědecky")
fig = px.sunburst(
    hierarchy_df,
    path=['cath_class', 'cath_arch', 'cath_topology', 'cath_homology'],
    values='len',
    color='cath_class',
    color_discrete_sequence=px.colors.qualitative.Safe # Barvy vhodné pro tisk
)

# Úprava vzhledu pro publikaci
fig.update_layout(
    margin=dict(t=10, l=10, r=10, b=10),
    font=dict(family="Arial", size=12),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

# 4. EXPORT DO PDF (Vektorový formát pro publikace)
fig.write_image("cath_distribution_publication.pdf", width=1000, height=1000)

print("Vektorové PDF bylo vytvořeno.")

Vektorové PDF bylo vytvořeno.


In [4]:
import polars as pl

# Předpokládáme, že tvůj dataframe se jmenuje 'df'

# 1. Celkové statistiky (Counts)
stats = df.select([
    pl.len().alias("Total_Sequences"),
    pl.col("cath_class").n_unique().alias("Unique_Classes"),
    pl.col("cath_arch").n_unique().alias("Unique_Architectures"),
    pl.col("cath_topology").n_unique().alias("Unique_Topologies"),
    pl.col("cath_homology").n_unique().alias("Unique_Superfamilies")
])

print("--- Základní přehled ---")
print(stats)

# 2. Distribuce podle CATH Class (nejčastější tabulka v publikaci)
class_dist = (
    df.group_by("cath_class")
    .agg([
        pl.len().alias("Count"),
        (pl.len() / len(df) * 100).round(2).alias("Percentage (%)"),
        pl.col("cath_homology").n_unique().alias("Num_Superfamilies")
    ])
    .sort("Count", descending=True)
)

print("\n--- Distribuce podle Tříd (C-level) ---")
print(class_dist)

# 3. Top 5 nejzastoupenějších Homologií (Superfamilies)
top_homologies = (
    df.group_by(["cath_class", "cath_arch", "cath_topology", "cath_homology"])
    .len()
    .sort("len", descending=True)
    .head(5)
    .rename({"len": "Sequence_Count"})
)

print("\n--- Top 5 nejčastějších Superrodin ---")
print(top_homologies)

# 4. Export do CSV (pro Excel/Tabulku v publikaci)
# class_dist.write_csv("cath_class_distribution.csv")

--- Základní přehled ---
shape: (1, 5)
┌─────────────────┬────────────────┬─────────────────────┬───────────────────┬─────────────────────┐
│ Total_Sequences ┆ Unique_Classes ┆ Unique_Architecture ┆ Unique_Topologies ┆ Unique_Superfamilie │
│ ---             ┆ ---            ┆ s                   ┆ ---               ┆ s                   │
│ u32             ┆ u32            ┆ ---                 ┆ u32               ┆ ---                 │
│                 ┆                ┆ u32                 ┆                   ┆ u32                 │
╞═════════════════╪════════════════╪═════════════════════╪═══════════════════╪═════════════════════╡
│ 864033          ┆ 5              ┆ 12                  ┆ 66                ┆ 44                  │
└─────────────────┴────────────────┴─────────────────────┴───────────────────┴─────────────────────┘

--- Distribuce podle Tříd (C-level) ---
shape: (5, 4)
┌────────────┬────────┬────────────────┬───────────────────┐
│ cath_class ┆ Count  ┆ Percentage (%

In [5]:
import polars as pl

# 1. Počet rodin (Unikátní CATH Superfamilies)
num_families = df.select(pl.col("cath_homology").n_unique()).item()

# 2. Úspěšnost klasifikace (pokud máš sloupec s predikcí, např. 'pred_cath')
# Pokud data už máš z CATH, úspěšnost je 100% (gold standard).
# Pokud jsi je predikoval:
# accuracy = (df.filter(pl.col("cath_dominant") == pl.col("pred_cath")).len() / len(df)) * 100

# 3. Sekvenční podobnost (v rámci rodin)
# Typicky se uvádí, zda jsi data "redunancy-reduced" (např. na 40% identity)
# Můžeme spočítat průměrnou délku sekvencí jako proxy pro komplexitu
avg_seq_len = df.select(pl.col("original_seq_full").str.len_chars().mean()).item()

# 4. Strukturální podobnost (CATH hierarchy depth)
# Kolik procent tvých dat sdílí stejnou Topologii?
top_topology = df.group_by("cath_topology").len().sort("len", descending=True).head(1)
top_topo_name = top_topology["cath_topology"][0]
top_topo_pct = (top_topology["len"][0] / len(df)) * 100

print(f"--- Hodnoty pro tabulku ---")
print(f"Počet rodin: {num_families}")
print(f"Průměrná délka sekvence: {avg_seq_len:.1f} AA")
print(f"Nejzastoupenější topologie: {top_topo_name} ({top_topo_pct:.1f} % datasetu)")

--- Hodnoty pro tabulku ---
Počet rodin: 44
Průměrná délka sekvence: 547.6 AA
Nejzastoupenější topologie: 30 (13.2 % datasetu)


In [6]:
import polars as pl
import pandas as pd

META_PARQUET = "../protbert/datasets/dataset_merged_with_families.parquet"

print("⏳ 1. Tvořím 'Slovník CATH anotací' ze VŠECH dat (sbírám všechny dostupné predikce)...")
df_cath_dict = (
    pl.scan_parquet(META_PARQUET)
    .select([
        pl.col("original_seq_full").alias("sequence"),
        "cath_class", "cath_arch", "cath_topology", "cath_homology"
    ])
    .drop_nulls(subset=["cath_class"])
    .unique(subset=["sequence"])
)

print("⏳ 2. Vybírám čisté páry (pouze reverse == False) a průměruji duplicitní měření...")
df_pairs = (
    pl.scan_parquet(META_PARQUET)
    .filter(pl.col("reverse") == False) # <--- Tady je ten tvůj geniální fix
    .select([
        pl.col("original_seq_full").alias("wt_seq"),
        pl.col("mutated_seq_full").alias("mut_seq"),
        "target"
    ])
    # Sloučíme případná vícenásobná měření stejné mutace do jednoho průměrného skóre
    .group_by(["wt_seq", "mut_seq"])
    .agg(pl.col("target").mean().alias("target"))
)

print("⏳ 3. Propojuji data pomocí LEFT JOIN...")
df_joined = df_pairs.join(
    df_cath_dict.select([
        pl.col("sequence").alias("wt_seq"),
        pl.col("cath_class").alias("wt_class"),
        pl.col("cath_arch").alias("wt_arch"),
        pl.col("cath_topology").alias("wt_topology"),
        pl.col("cath_homology").alias("wt_homology"),
    ]),
    on="wt_seq", how="left"
).join(
    df_cath_dict.select([
        pl.col("sequence").alias("mut_seq"),
        pl.col("cath_class").alias("mut_class"),
        pl.col("cath_arch").alias("mut_arch"),
        pl.col("cath_topology").alias("mut_topology"),
        pl.col("cath_homology").alias("mut_homology"),
    ]),
    on="mut_seq", how="left"
)

print("⏳ 4. Klasifikuji kompletní osudy a počítám průměrný fitness...")
df_analysis = (
    df_joined.with_columns([
        pl.when(pl.col("wt_class").is_not_null() & pl.col("mut_class").is_null())
        .then(pl.lit("1. Ztráta klasifikace (WT měl, Mutant nemá)"))

        .when(pl.col("wt_class").is_null() & pl.col("mut_class").is_not_null())
        .then(pl.lit("2. Zisk klasifikace (WT neměl, Mutant má)"))

        .when(pl.col("wt_class") != pl.col("mut_class"))
        .then(pl.lit("3. Změna hlavní Třídy (Class)"))

        .when(pl.col("wt_arch") != pl.col("mut_arch"))
        .then(pl.lit("4. Změna Architektury (Architecture)"))

        .when(pl.col("wt_topology") != pl.col("mut_topology"))
        .then(pl.lit("5. Změna Topologie (Topology)"))

        .when(pl.col("wt_homology") != pl.col("mut_homology"))
        .then(pl.lit("6. Změna Homologie (Homologous SF)"))

        .when(pl.col("wt_class").is_null() & pl.col("mut_class").is_null())
        .then(pl.lit("8. Trvale bez klasifikace (WT i Mutant)"))

        .otherwise(pl.lit("7. Beze změny (Stabilní CATH)"))
        .alias("Strukturální osud")
    ])
    .group_by("Strukturální osud")
    .agg([
        pl.len().alias("Počet mutantů"),
        pl.col("target").mean().round(3).alias("Průměrný target"),
        pl.col("target").median().round(3).alias("Medián target")
    ])
    .collect()
)

total_analyzed = df_analysis["Počet mutantů"].sum()

df_summary = (
    df_analysis
    .with_columns([
        (pl.col("Počet mutantů") / total_analyzed * 100).round(2).alias("Podíl (%)")
    ])
    .sort("Strukturální osud")
    .select(["Strukturální osud", "Počet mutantů", "Podíl (%)", "Průměrný target", "Medián target"])
)

print(f"\n✅ Hotovo! Celkem zpracováno přesně {total_analyzed:,} unikátních WT->MUT párů.")

display(df_summary.to_pandas().style.format({
    "Počet mutantů": "{:,}",
    "Podíl (%)": "{:.2f} %",
    "Průměrný target": "{:.3f}",
    "Medián target": "{:.3f}"
}).hide(axis="index"))

⏳ 1. Tvořím 'Slovník CATH anotací' ze VŠECH dat (sbírám všechny dostupné predikce)...
⏳ 2. Vybírám čisté páry (pouze reverse == False) a průměruji duplicitní měření...
⏳ 3. Propojuji data pomocí LEFT JOIN...
⏳ 4. Klasifikuji kompletní osudy a počítám průměrný fitness...

✅ Hotovo! Celkem zpracováno přesně 860,357 unikátních WT->MUT párů.


Strukturální osud,Počet mutantů,Podíl (%),Průměrný target,Medián target
"1. Ztráta klasifikace (WT měl, Mutant nemá)","88,357",10.27 %,-0.133,-0.097
3. Změna hlavní Třídy (Class),"518,004",60.21 %,-0.133,-0.097
4. Změna Architektury (Architecture),"130,894",15.21 %,-0.134,-0.097
5. Změna Topologie (Topology),"90,654",10.54 %,-0.131,-0.096
6. Změna Homologie (Homologous SF),"12,783",1.49 %,-0.143,-0.107
7. Beze změny (Stabilní CATH),"19,665",2.29 %,-0.137,-0.100


In [5]:
df_joined.describe()

statistic,true_wt_seq,true_mut_seq,target,wt_class,wt_arch,wt_topology,wt_homology,mut_class,mut_arch,mut_topology,mut_homology
str,str,str,f64,str,str,str,str,str,str,str,str
"""count""","""1736843""","""1736843""",1.736843e6,"""1637324""","""1637324""","""1637324""","""1637324""","""1648219""","""1648219""","""1648219""","""1648219"""
"""null_count""","""0""","""0""",0.0,"""99519""","""99519""","""99519""","""99519""","""88624""","""88624""","""88624""","""88624"""
"""mean""",null,null,0.052945,null,null,null,null,null,null,null,null
"""std""",null,null,0.297017,null,null,null,null,null,null,null,null
"""min""","""DTINITLPDGKTLTLTVTPEFTVKELAEEI…","""*DPFLVLLHSVSSSLSSSELTELKFLCLGR…",-0.986593,"""G3DSA:1""","""10""","""10""","""10""","""G3DSA:1""","""10""","""10""","""10"""
"""25%""",null,null,-0.109767,null,null,null,null,null,null,null,null
"""50%""",null,null,0.000983,null,null,null,null,null,null,null,null
"""75%""",null,null,0.182763,null,null,null,null,null,null,null,null
"""max""","""SVPQRAWTVEQLRSEQLPKKDIIKFLQEHG…","""YVIRSIIKSSRLEEDRKRYLMTLLDDIKGA…",0.999969,"""G3DSA:6""","""90""","""950""","""920""","""G3DSA:6""","""90""","""950""","""920"""


In [3]:
import polars as pl
import pandas as pd

# Načteme ten náš čerstvě vygenerovaný soubor!
NEW_DATASET = "../protbert/datasets/dataset_merged_with_families.parquet"

print("⏳ 1. Připravuji čisté dopředné páry a průměruji target...")
# Vytáhneme si základní páry z původního datasetu (pozor, musíš mít cestu k souboru, kde je 'target')
# Předpokládám, že 'target' a 'reverse' máš v dataset_merged.parquet nebo podobném
df_base = (
    pl.scan_parquet("../protbert/datasets/dataset_merged.parquet") # Změň cestu, pokud je jiná
    .filter(pl.col("reverse") == False)
    .select([
        pl.col("original_seq_full").alias("wt_seq"),
        pl.col("mutated_seq_full").alias("mut_seq"),
        "target"
    ])
    .group_by(["wt_seq", "mut_seq"])
    .agg(pl.col("target").mean().alias("target"))
)

print("⏳ 2. Načítám nové CATH anotace (cath_all)...")
df_cath = (
    pl.scan_parquet(NEW_DATASET)
    .select([
        pl.col("original_seq_full").alias("seq"),
        "cath_all"
    ])
    .unique()
)

print("⏳ 3. Propojuji a vyhodnocuji stabilitu...")
df_joined = (
    df_base
    .join(df_cath.rename({"seq": "wt_seq", "cath_all": "wt_cath"}), on="wt_seq", how="left")
    .join(df_cath.rename({"seq": "mut_seq", "cath_all": "mut_cath"}), on="mut_seq", how="left")
)

df_analysis = (
    df_joined.with_columns([
        pl.when(pl.col("wt_cath").is_not_null() & pl.col("mut_cath").is_null())
        .then(pl.lit("1. Ztráta klasifikace (WT měl, Mutant nemá)"))

        .when(pl.col("wt_cath").is_null() & pl.col("mut_cath").is_not_null())
        .then(pl.lit("2. Zisk klasifikace (WT neměl, Mutant má)"))

        .when(pl.col("wt_cath").is_null() & pl.col("mut_cath").is_null())
        .then(pl.lit("4. Trvale bez klasifikace (WT i Mutant)"))

        # TADY JE TA MAGIE: Porovnáváme přesné sety domén
        .when(pl.col("wt_cath") == pl.col("mut_cath"))
        .then(pl.lit("5. Beze změny (Absolutně stabilní domény)"))

        .when(pl.col("wt_cath") != pl.col("mut_cath"))
        .then(pl.lit("3. Změna složení domén (Skokan)"))

        .otherwise(pl.lit("Neznámý stav"))
        .alias("Osud")
    ])
    .group_by("Osud")
    .agg([
        pl.len().alias("Počet mutantů"),
        pl.col("target").mean().round(3).alias("Průměrný target"),
        pl.col("target").median().round(3).alias("Medián target")
    ])
    .collect()
)

# Finální procenta a výpis
total = df_analysis["Počet mutantů"].sum()
df_summary = (
    df_analysis
    .with_columns([(pl.col("Počet mutantů") / total * 100).round(2).alias("Podíl (%)")])
    .sort("Osud")
    .select(["Osud", "Počet mutantů", "Podíl (%)", "Průměrný target", "Medián target"])
)

print(f"✅ Hotovo! Analyzováno {total:,} párů.")
display(df_summary.to_pandas().style.format({
    "Počet mutantů": "{:,}",
    "Podíl (%)": "{:.2f} %",
    "Průměrný target": "{:.3f}",
    "Medián target": "{:.3f}"
}).hide(axis="index"))

⏳ 1. Připravuji čisté dopředné páry a průměruji target...
⏳ 2. Načítám nové CATH anotace (cath_all)...
⏳ 3. Propojuji a vyhodnocuji stabilitu...
✅ Hotovo! Analyzováno 970,366 párů.


Osud,Počet mutantů,Podíl (%),Průměrný target,Medián target
"1. Ztráta klasifikace (WT měl, Mutant nemá)","88,357",9.11 %,-0.133,-0.097
"2. Zisk klasifikace (WT neměl, Mutant má)","98,730",10.17 %,-0.142,-0.099
3. Změna složení domén (Skokan),"758,860",78.20 %,-0.133,-0.097
4. Trvale bez klasifikace (WT i Mutant),"11,279",1.16 %,-0.142,-0.097
5. Beze změny (Absolutně stabilní domény),"13,140",1.35 %,-0.138,-0.103
